# Validação da base processada

Este notebook audita a base processada existente e a compara com a planilha original de 2024. **Nenhuma base é alterada ou sobrescrita.**

## 1. Configuração e caminhos

Os caminhos são resolvidos a partir da raiz do projeto para permitir execução local ou a partir da pasta `notebooks`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

RAIZ = Path.cwd()
if not (RAIZ / 'src').exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

from src.validacao_dados import (
    carregar_tabela, comparar_bases, estatisticas_indicadores,
    perfil_colunas, validar_dataframe
)

CAMINHO_ORIGINAL = RAIZ / 'data/raw/BASE DE DADOS PEDE 2024 - DATATHON.xlsx'
CAMINHO_PROCESSADA = RAIZ / 'data/processed/base_2024_clean.csv'
PLANILHA = 'PEDE2024'
CAMINHO_ORIGINAL.exists(), CAMINHO_PROCESSADA.exists()

## 2. Carga somente leitura

In [ ]:
df_original = carregar_tabela(CAMINHO_ORIGINAL, planilha=PLANILHA)
df_processada = carregar_tabela(CAMINHO_PROCESSADA)

pd.DataFrame({
    'base': ['Original PEDE2024', 'Processada local'],
    'linhas': [len(df_original), len(df_processada)],
    'colunas': [df_original.shape[1], df_processada.shape[1]],
    'duplicidades integrais': [df_original.duplicated().sum(), df_processada.duplicated().sum()]
})

## 3. Perfil de colunas, tipos e nulos

In [ ]:
perfil_processada = perfil_colunas(df_processada)
perfil_processada.sort_values(['percentual_nulos', 'coluna'], ascending=[False, True])

## 4. Estatísticas e faixas dos indicadores

In [ ]:
estatisticas_indicadores(df_processada)

## 5. Valores categóricos relevantes e identificadores

In [ ]:
colunas_categoricas = [
    c for c in ['fase', 'pedra', 'genero', 'instituicao_de_ensino',
                'fase_ideal', 'ativo_inativo', 'defasagem', 'ian']
    if c in df_processada.columns
]
valores_relevantes = {c: df_processada[c].value_counts(dropna=False).to_dict() for c in colunas_categoricas}
valores_relevantes

In [ ]:
pd.DataFrame({
    'verificação': ['RAs únicos', 'RAs duplicados', 'linhas duplicadas', 'IAA acima de 10'],
    'resultado': [
        df_processada['ra'].nunique(),
        df_processada.duplicated('ra').sum(),
        df_processada.duplicated().sum(),
        (pd.to_numeric(df_processada['iaa'], errors='coerce') > 10).sum()
    ]
})

## 6. Comparação entre original e processada

In [ ]:
comparacao = comparar_bases(df_original, df_processada)
{chave: comparacao[chave] for chave in [
    'linhas_antes', 'linhas_depois', 'diferenca_linhas',
    'colunas_antes', 'colunas_depois', 'colunas_removidas', 'colunas_criadas'
]}

In [ ]:
pd.DataFrame(comparacao['tipos_alterados'])

## 7. Conclusão da auditoria

A base local processada preserva as 1.156 linhas de 2024, mas não substitui a base histórica consolidada mencionada na EDA. Valores `INCLUIR`, IAA igual a 10,002, ausências estruturais e colunas constantes devem ser documentados antes de qualquer correção. O significado do target e a data de disponibilidade dos indicadores permanecem pendentes.